# DIP + BATDiff on the penguin photograph

I use this notebook to train BATDiff on the full DIV2K penguin photograph, with my DIP reconstruction as the reference image `x_ref` (`--xref_image`). I do not train DIP again here. I already have `dip.png` from the DIP-only run.

This is not BATDiff alone. BATDiff alone used a bicubic enlargement of `lr.png` as `x_ref`. Here I swap that reference for DIP and train to the paper length of **120000 steps**.

A Colab T4 cannot hold this full photograph at `dim=200`, so I keep `dim=64`. I already finished 20000 steps (`model-1.pt`). Later sessions continue from the newest `model-N.pt` on Google Drive until 120000. Checkpoints are written every 10000 steps.

I do not use `batdiff_full_colab.ipynb` for this run. That notebook is BATDiff alone on a 256 crop, with no DIP reference.

| Step | What I do | What I expect |
|---|---|---|
| **0** | Check that Colab has a GPU | `Tesla T4` |
| **1** | Clone BATDiff and my project | no error |
| **2** | Patch BATDiff so `--xref_image` works | `Patch applied` |
| **3** | Upload `lr.png`, `hr.png`, and `dip.png` | LR 510x339, HR 2040x1356, and a penguin preview |
| **4** | Mount Drive and keep the DIP + 120k switches | `DIP+BATDiff` and `found checkpoint` |
| **5** | Print the BATDiff flags | `train steps 120000` and `xref DIP` |
| **6** | Load the runner | `Runner ready` |
| **7** | Train | a `step:` line, then many hours |
| **8–9** | Score and figure | only after Step 7 prints `training completed` |

I set the runtime to **T4 GPU** in Chrome. I upload this file from `notebooks/batdiff_natural.ipynb`. The three images are in `outputs/sanity/div2k_filtered_stride_x4/`. `dip.png` is the DIP reconstruction, not a BATDiff sample. Checkpoints stay on Drive under `batdiff_penguin_dip_xref`.


In [ ]:
#@title Step 0 — check the runtime has a GPU
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU visible.\n"
        "Fix: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, "
        "then run this cell again."
    )

total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU:   {torch.cuda.get_device_name(0)}")
print(f"VRAM:  {total_gb:.1f} GB")
print(f"torch: {torch.__version__}")
if "T4" not in torch.cuda.get_device_name(0) and total_gb < 14:
    print("Warning: this is not a T4. The full photo may run out of memory.")

## Step 1 — Clone BATDiff and my project

I run the next cell. Nothing to upload here.


In [ ]:
#@title Step 1 — clone repos and install four packages
import os, subprocess
from pathlib import Path

PROJECT_REPO = "https://github.com/yoyowuyogwrt-hue/3D-OCT-Image-SuperResolution-Benchmark"
BATDIFF_REPO = "https://github.com/MaryamHeidari-1994/BATDiff"

WORK     = Path("/content")
BATDIFF  = WORK / "BATDiff"
PROJECT  = WORK / "project"


def sh(cmd, cwd=None):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(f"command failed with exit code {result.returncode}")


if not BATDIFF.exists():
    sh(f"git clone --depth 1 {BATDIFF_REPO} {BATDIFF}")
if not PROJECT.exists():
    sh(f"git clone --depth 1 {PROJECT_REPO} {PROJECT}")

sh("pip install -q einops ftfy regex PyWavelets lpips")
print("\nBATDiff:", BATDIFF)
print("project:", PROJECT)

In [ ]:
#@title Step 2 — patch BATDiff so --xref_image works
sh(f"python {PROJECT}/scripts/batdiff_dip_patch.py --batdiff-root {BATDIFF}")
print("Patch applied. Step 4 USE_DIP_XREF decides whether --xref_image is passed.")

## Step 3 — Upload the penguin pair and the DIP reference

I run the next cell and choose three files from

`outputs/sanity/div2k_filtered_stride_x4/`

| File | What it is |
|---|---|
| `lr.png` | low-resolution penguin (blur then stride x4) |
| `hr.png` | original photograph, scoring only |
| `dip.png` | my DIP reconstruction; this becomes BATDiff `x_ref` |

If Colab kept only one file, I run the cell again and add the missing ones. I should see `LR 510x339`, `HR 2040x1356`, `saved dip.png`, and a penguin picture.


In [ ]:
#@title Step 3 — upload lr.png, hr.png, and dip.png
from PIL import Image
from google.colab import files
from IPython.display import display

SR_FACTOR = 4
DATA = WORK / "data" / "natural"
DATA.mkdir(parents=True, exist_ok=True)
HR_SRC = WORK / "hr.png"

print("Choose lr.png, hr.png, and dip.png (hold Cmd to select all three).")
uploaded = files.upload()

for name, data in uploaded.items():
    lower = name.lower()
    if lower == "lr.png":
        (DATA / "lr.png").write_bytes(data)
        print("saved lr.png")
    elif lower in ("hr.png", "hr_crop.png"):
        HR_SRC.write_bytes(data)
        (DATA / "hr.png").write_bytes(data)
        (DATA / "hr_crop.png").write_bytes(data)
        print("saved hr.png")
    elif lower == "dip.png":
        (DATA / "dip.png").write_bytes(data)
        print("saved dip.png  (this will be BATDiff x_ref)")
    else:
        print(f"ignored extra file: {name}")

if not (DATA / "lr.png").exists() or not HR_SRC.exists() or not (DATA / "dip.png").exists():
    missing = [n for n in ("lr.png", "hr.png", "dip.png") if not (DATA / n).exists() and not (HR_SRC.exists() and n == "hr.png")]
    if not HR_SRC.exists():
        missing.append("hr.png")
    if not (DATA / "dip.png").exists():
        missing.append("dip.png")
    raise FileNotFoundError(
        "Need lr.png, hr.png, and dip.png.\n"
        f"Still missing: {sorted(set(missing))}\n"
        "Run this cell again and upload the missing file(s)."
    )

lr_size = Image.open(DATA / "lr.png").size
hr_size = Image.open(HR_SRC).size
dip_size = Image.open(DATA / "dip.png").size
expected_hr = (lr_size[0] * SR_FACTOR, lr_size[1] * SR_FACTOR)
if hr_size != expected_hr:
    raise ValueError(
        f"HR is {hr_size}, but LR {lr_size} at x{SR_FACTOR} expects {expected_hr}.\n"
        "I should use outputs/sanity/div2k_filtered_stride_x4/ on this Mac."
    )
if dip_size != hr_size:
    raise ValueError(f"dip.png is {dip_size}, but hr.png is {hr_size}. Upload the matching pair.")

print(f"LR  {lr_size[0]}x{lr_size[1]}")
print(f"HR  {hr_size[0]}x{hr_size[1]}  (scoring only)")
print(f"DIP {dip_size[0]}x{dip_size[1]}  -> will be --xref_image {DATA / 'dip.png'}")
print("Preview of uploaded DIP (should look like a penguin, not snow):")
display(Image.open(DATA / "dip.png"))

## Step 4 — Switches and Google Drive

For the paper-length DIP+BATDiff run I keep:

| Box | Setting |
|---|---|
| `SMOKE_TEST` | off |
| `USE_DIP_XREF` | on |
| `SAVE_TO_DRIVE` | on |
| `FULL_120K` | on |
| `CONTINUE_FROM_20K` | on |

`LEAVE_12H` is unused when `FULL_120K` is on. Chrome asks to connect Drive; I click Allow.

I look for `DIP+BATDiff`, `--xref_image`, `found checkpoint`, and `Mounted at /content/drive`. The checkpoint name is the newest `model-N.pt` already on Drive, not a PNG.


In [ ]:
#@title Step 4 — today's switches
SMOKE_TEST = False  #@param {type:"boolean"}
USE_DIP_XREF = True  #@param {type:"boolean"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
FULL_120K = True  #@param {type:"boolean"}
CONTINUE_FROM_20K = True  #@param {type:"boolean"}
LEAVE_12H = True  #@param {type:"boolean"}

import re
from google.colab import drive

if USE_DIP_XREF and not (DATA / "dip.png").exists():
    raise FileNotFoundError(
        "USE_DIP_XREF is on but dip.png is missing. Go back to Step 3 and upload it."
    )

if SAVE_TO_DRIVE:
    drive.mount("/content/drive")
    RESULTS = Path("/content/drive/MyDrive/batdiff_penguin_dip_xref")
    print("Results will be saved to Google Drive:", RESULTS)
else:
    RESULTS = WORK / "results"
    print("Results stay on this Colab machine only. Download the zip in Step 7 before the runtime dies.")
RESULTS.mkdir(parents=True, exist_ok=True)

print("SMOKE TEST" if SMOKE_TEST else ("DIP+BATDiff" if USE_DIP_XREF else "BATDiff alone"))
if USE_DIP_XREF:
    print(f"will pass --xref_image {DATA / 'dip.png'}")
    print("Colab will NOT re-train DIP.")
else:
    print("no --xref_image (published bicubic x_ref)")

tag_guess = "dip_xref" if USE_DIP_XREF else "natural"
ckpt_dir = RESULTS / tag_guess / tag_guess


def newest_milestone(folder: Path) -> int:
    best = 0
    if not folder.exists():
        return 0
    for path in folder.glob("model-*.pt"):
        match = re.search(r"model-(\d+)\.pt$", path.name)
        if match:
            best = max(best, int(match.group(1)))
    return best


FOUND_MILESTONE = newest_milestone(ckpt_dir)

if SMOKE_TEST:
    print("SMOKE_TEST is on. Uncheck it for the real run.")
elif FULL_120K and CONTINUE_FROM_20K:
    if FOUND_MILESTONE < 1:
        raise FileNotFoundError(
            "CONTINUE_FROM_20K is on, but Drive has no model-N.pt at\n"
            f"  {ckpt_dir}\n"
            "Mount the same folder as today's run: My Drive / batdiff_penguin_dip_xref"
        )
    print(f"FULL 120k: continue from model-{FOUND_MILESTONE}.pt toward 120000")
    print("Remaining on a T4: about 30-36 hours from 20k. Save every 10000 steps to Drive.")
    print("found checkpoint", ckpt_dir / f"model-{FOUND_MILESTONE}.pt")
elif FULL_120K:
    print("FULL 120k from step 0 (about 36 hours). You will throw away today's 20k.")
elif LEAVE_12H:
    print("LEAVE 12H: 20000 steps (~9 hours on a T4). Sampling at the end needs extra time.")
else:
    print("Short comparable run: 4000 steps (~2 hours). Too short to leave for 12 hours.")


## Step 5 — BATDiff configuration

I run this cell and check the printed flags: `train steps 120000`, `dim 64`, `sr_factor 4`, `xref DIP`. I leave `dim` at 64 because a T4 cannot fit this full photograph at 128 or 200.


In [ ]:
#@title Step 5 — BATDiff configuration
if SMOKE_TEST:
    DIM, TRAIN_STEPS, TIMESTEPS, SAVE_EVERY, LOAD_MILESTONE = 16, 60, 20, 60, 0
elif FULL_120K:
    DIM, TRAIN_STEPS, TIMESTEPS, SAVE_EVERY = 64, 120000, 100, 10000
    LOAD_MILESTONE = FOUND_MILESTONE if CONTINUE_FROM_20K else 0
elif LEAVE_12H:
    DIM, TRAIN_STEPS, TIMESTEPS, SAVE_EVERY, LOAD_MILESTONE = 64, 20000, 100, 20000, 0
else:
    DIM, TRAIN_STEPS, TIMESTEPS, SAVE_EVERY, LOAD_MILESTONE = 64, 4000, 100, 4000, 0

ATROUS_LEVEL = 6
XREF_PATH = DATA / "dip.png" if USE_DIP_XREF else None

COMMON_FLAGS = (
    f"--mode train "
    f"--image_name lr.png "
    f"--use_atrous --atrous_wavelet b3 "
    f"--atrous_level {ATROUS_LEVEL} "
    f"--sr_factor {SR_FACTOR} "
    f"--dim {DIM} "
    f"--ts 1 "
    f"--train_num_steps {TRAIN_STEPS} "
    f"--timesteps {TIMESTEPS} "
    f"--save_and_sample_every {SAVE_EVERY} "
)
if LOAD_MILESTONE:
    COMMON_FLAGS += f"--load_milestone {LOAD_MILESTONE} "

print("SMOKE TEST" if SMOKE_TEST else "REAL RUN")
print(f"  dim         {DIM}")
print(f"  ts          1")
print(f"  train steps {TRAIN_STEPS}")
print(f"  save every  {SAVE_EVERY}")
print(f"  load        {LOAD_MILESTONE}")
print(f"  sr_factor   {SR_FACTOR}")
print(f"  xref        {'DIP ' + str(XREF_PATH) if XREF_PATH else 'bicubic upsample of LR'}")
print(f"  results     {RESULTS}")
if not SMOKE_TEST:
    already = 20000 if (LOAD_MILESTONE == 1) else (LOAD_MILESTONE * SAVE_EVERY if LOAD_MILESTONE else 0)
    if LOAD_MILESTONE == 1:
        already = 20000  # today's file was saved every 20000 steps
    remaining = max(TRAIN_STEPS - already, 0)
    hours = remaining / 3200
    print(f"  already     ~{already} steps")
    print(f"  remaining   ~{hours:.0f} hours on a T4")


In [ ]:
#@title Step 6 — BATDiff runner (press play; this does not start training yet)
import re, sys, time, subprocess
from pathlib import Path


def find_final_image(scope_dir: Path) -> Path:
    candidates = list((scope_dir / "final_samples").glob("*.png"))
    if not candidates:
        raise FileNotFoundError(f"No samples under {scope_dir / 'final_samples'}")

    def scale_of(path: Path) -> int:
        match = re.search(r"_s(\d+)_", path.name)
        return int(match.group(1)) if match else -1

    finest = max(scale_of(p) for p in candidates)
    at_finest = [p for p in candidates if scale_of(p) == finest]
    return max(at_finest, key=lambda p: p.stat().st_mtime)


def run_batdiff(tag: str, dataset_folder: Path, xref: Path | None) -> Path:
    scope_dir = RESULTS / tag / tag
    scope_dir.mkdir(parents=True, exist_ok=True)
    flags = COMMON_FLAGS + f"--scope {tag} "
    flags += f"--dataset_folder {dataset_folder}/ --results_folder {RESULTS / tag} "
    if xref is not None:
        flags += f"--xref_image {xref} "

    print("=" * 70)
    print(f"RUN {tag}   reference = {'DIP output' if xref else 'bicubic (upstream)'}")
    print(f"flags: {flags}")
    print("=" * 70)

    started = time.time()
    process = subprocess.Popen(
        f"PYTHONUNBUFFERED=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u main.py {flags}",
        shell=True, cwd=BATDIFF, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    for line in process.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    process.wait()

    elapsed = (time.time() - started) / 60
    if process.returncode:
        raise RuntimeError(f"run {tag} failed with exit code {process.returncode}")

    output = find_final_image(scope_dir)
    print(f"\nfinished in {elapsed:.1f} min -> {output.name}")
    return output


print("Runner ready. Next cell (Step 7) starts the long training.")
if LOAD_MILESTONE:
    print("Leave only after you see a step number at or above 20000.")
else:
    print("Leave only after you see step:100.")


## Step 7 — Train BATDiff

I run this cell once. Training takes many hours. The first `step:` line can take several minutes. If I loaded a checkpoint, the step number should match that file (for example 80000 after `model-8.pt`), not 0.

I score in Step 8 only after this cell prints `training completed`. If Colab stops early, Drive should already hold `model-N.pt` every 10000 steps, and I continue from the newest file.


In [ ]:
#@title Step 7 — run BATDiff (this is the long cell — leave after a step: line)
import shutil
from google.colab import files

tag = "dip_xref" if USE_DIP_XREF else "natural"
out_batdiff = run_batdiff(tag, DATA, xref=XREF_PATH)

archive_stem = WORK / "batdiff_penguin_dip_xref_results"
archive = shutil.make_archive(str(archive_stem), "zip", RESULTS)
print("saved zip:", archive)

if SAVE_TO_DRIVE:
    drive_zip = RESULTS / "batdiff_penguin_dip_xref_results.zip"
    shutil.copy2(archive, drive_zip)
    print("copied zip to Drive:", drive_zip)

try:
    files.download(archive)
except Exception as error:
    print(f"(browser download unavailable: {error})")
    print("Use Google Drive folder batdiff_penguin_dip_xref, or the Files pane on the left.")

print("\nCome back: run Step 8 then Step 9 if this cell reached here with no red error.")


## Step 8 — Scores

I run this after Step 7 prints `training completed`.


In [ ]:
#@title Step 8 — score against the original photograph
import sys, csv
import numpy as np
from PIL import Image

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.metrics.evaluate import evaluate

hr = np.asarray(Image.open(HR_SRC).convert("RGB"))
target_size = (hr.shape[1], hr.shape[0])
lr = Image.open(DATA / "lr.png").convert("RGB")
bicubic = np.asarray(lr.resize(target_size, Image.BICUBIC))


def load_like_hr(path):
    image = Image.open(path).convert("RGB")
    if image.size != target_size:
        print(f"note: resizing {path.name} from {image.size} to {target_size}")
        image = image.resize(target_size, Image.BICUBIC)
    return np.asarray(image)


methods = {
    "Bicubic upsample": bicubic,
    "DIP alone":        load_like_hr(DATA / "dip.png"),
    "BATDiff (DIP ref)": load_like_hr(out_batdiff),
}

rows = []
for name, image in methods.items():
    scores = evaluate(hr, image)
    rows.append({"method": name, **scores})

print(f"\n{'method':<22}{'PSNR ↑':>9}{'SSIM ↑':>9}{'LPIPS ↓':>10}")
print("-" * 50)
for row in rows:
    print(f"{row['method']:<22}{row['PSNR']:>9.4f}{row['SSIM']:>9.4f}{row['LPIPS']:>10.4f}")

out_csv = RESULTS / ("scores_smoke.csv" if SMOKE_TEST else "scores.csv")
with out_csv.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["method", "PSNR", "SSIM", "LPIPS"])
    writer.writeheader()
    writer.writerows(rows)
print(f"\nsaved {out_csv}")
if SMOKE_TEST:
    print("SMOKE TEST numbers are meaningless.")

In [ ]:
#@title Step 9 — comparison figure and second zip
import matplotlib.pyplot as plt

panels = [("HR (ground truth)", hr)] + [
    (row["method"], methods[row["method"]]) for row in rows
]

fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.2))
for axis, (title, image) in zip(axes, panels):
    axis.imshow(image)
    axis.set_title(title, fontsize=11)
    axis.axis("off")
    if title != "HR (ground truth)":
        row = next(r for r in rows if r["method"] == title)
        axis.text(
            0.5, -0.04,
            f"PSNR {row['PSNR']:.2f}  SSIM {row['SSIM']:.3f}  LPIPS {row['LPIPS']:.3f}",
            transform=axis.transAxes, ha="center", va="top", fontsize=8,
        )
fig.tight_layout()

figure_path = RESULTS / ("comparison_smoke.png" if SMOKE_TEST else "comparison.png")
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved {figure_path}")

archive = shutil.make_archive(str(WORK / "batdiff_penguin_dip_xref_results"), "zip", RESULTS)
print(f"archive: {archive}")
if SAVE_TO_DRIVE:
    shutil.copy2(archive, RESULTS / "batdiff_penguin_dip_xref_results.zip")
try:
    files.download(archive)
except Exception as error:
    print(f"(browser download unavailable: {error})")

## If something fails

**No GPU** — I set Runtime → Change runtime type → T4 GPU, then run Step 0 again.

**Drive permission** — I click Allow. Chrome is more reliable than Safari.

**CUDA out of memory** — the full photograph already uses `DIM = 64`. I can drop it to 32, then run Step 5 and Step 7 again.

**Only one PNG uploaded** — I run Step 3 again and add the missing file.

**No checkpoint on Drive** — I mount the same folder as the earlier run, `My Drive / batdiff_penguin_dip_xref`.

**Colab disconnected** — I look for the newest `model-N.pt` on Drive, start a new T4, and run from Step 0 with the same three PNGs.

I copy finished zips into `outputs/sanity/div2k_filtered_stride_x4/batdiff/dip_xref/` on my Mac.
